# Synthesis and Capture

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import time

from acadia.system import Acadia, StreamConfiguration
from acadia.channel import Channel
from acadia.arrays import ProceduralWaveform, Waveform
from acadia.data import DataManager, ArrayRecordGroup

import logging
logging.basicConfig(format='[%(asctime)s] %(threadName)s: %(message)s', datefmt='%m/%d/%Y %I:%M:%S %p', level=logging.INFO)

# System Configuration

In [2]:
acadia = Acadia()

pulse_channel = acadia.DAC(1)

def pulse_shape(out, sample_times):
    out[:] = Channel.to_samples(0.99*np.ones(len(sample_times), dtype=np.complex64))

pulse = ProceduralWaveform(pulse_channel, pulse_shape, pulse_channel)

capture_channel = acadia.ADC(1)
capture_data = Waveform(capture_channel, 5000e-9, region=acadia.PLDDR0Array)
capture_configuration = StreamConfiguration(capture_channel, acadia=acadia)

def configure():
    pulse_channel.set_nyquist_zone(2)
    pulse_channel.configure_nco(frequency=2000e6)
    pulse_channel.set_vop(20000)
    
    capture_channel.set_nyquist_zone(2)
    capture_channel.set_dsa(0)
    
SHOTS = 1000
    
# We'll collect the data traces in a record group
traces = ArrayRecordGroup((len(capture_data),), 
                          SHOTS, 
                          dtype=np.complex64, 
                          record_axes=[capture_data.axis()])
    
# Make a data manager for storing data and serving it to a plotter
mgr = DataManager("/home/root", save_count=10)

def last_record(data):
    return {"records": np.array([np.mean(data["records"], axis=0)]), "axis0": data["axis0"]}

mgr.add_group("traces", traces, preprocessor=last_record)

## The actual meat of the program

In [3]:
arr = acadia.CacheArray(size=16)

# Create a sequence for the sequencer
def sequence(a):
    capture_configuration.reset()
    for i in range(100):
        a._active_sequencer.nop()
        
    with a.channel_synchronizer():
        a.generate(pulse_channel, pulse)
        a.capture(capture_configuration, capture_data)

# Because the pulse is procedurally generated, we can change its length at runtime,
# so we need to start by allocating the length to use initially
pulse.allocate(1000e-9)

# Attach to the hardware
acadia.attach()

# Load the wave memory with the pulse by calling the generator function
pulse.populate()

# Configure channel parameters using the function we defined above
configure()

# Configure the stream processing path to capture data using the configuration
# written above
acadia.configure_stream(capture_configuration)

# Compile only once
acadia.compile(sequence)

In [8]:
mgr.start_server()

traces._count = 0

from tqdm import tqdm

for shot in tqdm(range(SHOTS)):
    acadia.run(assemble=(shot==0))
    
    # Get the trace data and write it into a record
    trace = Channel.from_samples(capture_data.memory())
    mgr.append("traces", trace)
    
    # Wait some time until running again just so that we can see the plot update
    time.sleep(0.1)
    
mgr.stop_server()

100%|██████████| 1000/1000 [01:58<00:00,  5.85it/s]


In [5]:
pulse.memory()[:16]

array([32440,     0, 32440,     0, 32440,     0, 32440,     0, 32440,
           0, 32440,     0, 32440,     0, 32440,     0], dtype=int16)

In [6]:
acadia._dac_dma_descriptor_memory[1][:16]

array([249,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0], dtype=uint8)

In [9]:
acadia.sequencer_pprint()

---- Program 0 ----
0000: Symbol(assigned=True, value=0x1C8000) -> BUS_ADDR  |  00000000 -> BUS_DATA
0001: 00130003 -> BUS_ADDR  |  00000000 -> BUS_DATA
0002: NOP  |  NOP
0003: NOP  |  NOP
0004: NOP  |  NOP
0005: NOP  |  NOP
0006: NOP  |  NOP
0007: NOP  |  NOP
0008: NOP  |  NOP
0009: NOP  |  NOP
000A: NOP  |  NOP
000B: NOP  |  NOP
000C: NOP  |  NOP
000D: NOP  |  NOP
000E: NOP  |  NOP
000F: NOP  |  NOP
0010: NOP  |  NOP
0011: NOP  |  NOP
0012: NOP  |  NOP
0013: NOP  |  NOP
0014: NOP  |  NOP
0015: NOP  |  NOP
0016: NOP  |  NOP
0017: NOP  |  NOP
0018: NOP  |  NOP
0019: NOP  |  NOP
001A: NOP  |  NOP
001B: NOP  |  NOP
001C: NOP  |  NOP
001D: NOP  |  NOP
001E: NOP  |  NOP
001F: NOP  |  NOP
0020: NOP  |  NOP
0021: NOP  |  NOP
0022: NOP  |  NOP
0023: NOP  |  NOP
0024: NOP  |  NOP
0025: NOP  |  NOP
0026: NOP  |  NOP
0027: NOP  |  NOP
0028: NOP  |  NOP
0029: NOP  |  NOP
002A: NOP  |  NOP
002B: NOP  |  NOP
002C: NOP  |  NOP
002D: NOP  |  NOP
002E: NOP  |  NOP
002F: NOP  |  NOP
0030: NOP  |  NOP
0